# 놀이다리 로컬 AI 학습 (Colab)

놀이 기록 정리 → 센터 상담용 요약(JSON)을 쓰는 작은 모델을 만듭니다.
- 기본 모델: **Qwen2.5-3B-Instruct** (한국어 가능, 노트북에서 빠르게 동작)
- 방법: QLoRA (Unsloth), 무료 T4 GPU로 약 10~20분
- 결과: `noridari.Q4_K_M.gguf` → 내 컴퓨터 Ollama에서 사용

**준비**: 상단 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU` 선택 후 위에서부터 차례로 실행하세요.
⚠️ 학습 데이터는 가짜 기록으로 만든 것만 사용합니다. 실제 아이 정보를 올리지 마세요.

In [ ]:
# 1. 설치 (2~3분)
!pip install -q unsloth

In [ ]:
# 2. 학습 데이터 올리기: ai/data/train.jsonl 파일을 선택하세요
from google.colab import files
up = files.upload()
DATA = next(iter(up))
print(DATA)

In [ ]:
# 3. 기본 모델 불러오기
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/Qwen2.5-3B-Instruct", max_seq_length=2048, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0, bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=42)

In [ ]:
# 4. 데이터 준비 (10%는 시험용으로 빼둠)
from datasets import load_dataset
ds = load_dataset("json", data_files=DATA, split="train").train_test_split(test_size=0.1, seed=42)
fmt = lambda b: {"text": [tokenizer.apply_chat_template(m, tokenize=False) for m in b["messages"]]}
train = ds["train"].map(fmt, batched=True)
test = ds["test"]
print(len(train), "학습 /", len(test), "시험")

In [ ]:
# 5. 학습
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model=model, processing_class=tokenizer, train_dataset=train,
    args=SFTConfig(dataset_text_field="text", max_seq_length=2048,
                   per_device_train_batch_size=2, gradient_accumulation_steps=4,
                   num_train_epochs=3, learning_rate=2e-4, logging_steps=5,
                   optim="adamw_8bit", output_dir="out", seed=42, report_to="none"))
trainer.train()

In [ ]:
# 6. 시험: 학습에 안 쓴 기록으로 결과 확인 + 금지 표현 검사
import json, re
FastLanguageModel.for_inference(model)
BANNED = re.compile(r"진단|지연|향상|좋아졌|발달|점차|늘어|효과|적응|변화|성장|능력|수준")
bad = 0
for ex in test.select(range(min(5, len(test)))):
    msgs = ex["messages"][:2]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = tokenizer.decode(model.generate(ids, max_new_tokens=400, temperature=0.3)[0][ids.shape[1]:], skip_special_tokens=True)
    ok_json = True
    try: json.loads(out)
    except Exception: ok_json = False
    flag = (not ok_json) or bool(BANNED.search(out))
    bad += flag
    print("❌" if flag else "✅", out)
    print()
print(f"문제 {bad}개 / 5개")

In [ ]:
# 7. GGUF로 내보내서 구글 드라이브에 저장 (5~10분)
from google.colab import drive
drive.mount("/content/drive")
model.save_pretrained_gguf("noridari", tokenizer, quantization_method="q4_k_m")
!ls -la noridari_gguf
!mkdir -p /content/drive/MyDrive/noridari && cp noridari_gguf/*Q4_K_M.gguf /content/drive/MyDrive/noridari/noridari.Q4_K_M.gguf
print("드라이브 > noridari 폴더에서 noridari.Q4_K_M.gguf를 내려받아 ai/ 폴더에 넣으세요")

## 내 컴퓨터에서 쓰기
```
ollama create noridari -f ai/Modelfile
ollama run noridari
```